# Client Testing Notebook
Test each external client independently: Google Civic, Tavily, and Anthropic.

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv("../.env")

print("GOOGLE_CIVIC_API_KEY:", "set" if os.environ.get("GOOGLE_CIVIC_API_KEY") else "MISSING")
print("TAVILY_API_KEY:", "set" if os.environ.get("TAVILY_API_KEY") else "MISSING")
print("ANTHROPIC_API_KEY:", "set" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")

## 1. Google Civic Information API

In [ ]:
ELECTIONS_URL = "https://www.googleapis.com/civicinfo/v2/elections"

resp_elections = httpx.get(
    ELECTIONS_URL,
    params={"key": os.environ["GOOGLE_CIVIC_API_KEY"]},
    timeout=15,
)

print(f"Status: {resp_elections.status_code}")
if resp_elections.status_code != 200:
    print(f"Error: {resp_elections.text}")
    print()
    print("If this also returns 404 'Method not found', the Google Civic Information API")
    print("is NOT enabled in your GCP project. Go to:")
    print("  https://console.cloud.google.com/apis/library/civicinfo.googleapis.com")
    print("and click 'Enable'.")
else:
    import json
    data = resp_elections.json()
    print(f"Elections found: {len(data.get('elections', []))}")
    print(json.dumps(data, indent=2))

In [ ]:
# Inspect raw response for debugging
if resp.status_code == 200:
    import json
    print(json.dumps(resp.json(), indent=2))

## 2. Tavily Search API

In [ ]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

results = tavily.search(query="US Senator from California", max_results=3)

print(f"Results: {len(results.get('results', []))}")
for r in results.get("results", []):
    print(f"  - {r['title']}")
    print(f"    {r['url']}")
    print(f"    {r['content'][:150]}...")
    print()

## 3. Anthropic (Claude) API

In [ ]:
import anthropic

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)

print(f"Stop reason: {message.stop_reason}")
print(f"Response: {message.content[0].text}")